In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "README.md").is_file() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").is_file():
    raise FileNotFoundError("Run this notebook from the project directory or one of its subdirectories.")


In [1]:
# ============================================================
# 1. BASE CLEANING & CAUSAL IMPUTATION (V6 - NO LEAKAGE)
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

RAW_ATMOS_PATH = Path(PROJECT_ROOT / "data" / "raw" / "train_atmos.csv")
RAW_WAVE_PATH = Path(PROJECT_ROOT / "data" / "raw" / "train_wave.csv")
BASE_V6_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_base_v6.csv")

TIME_COL = "time"
STATION_COL = "station"
STEP_MINUTES = 10

BASE_COLUMNS = ["hs", "tp", "hmax", "wvdir", "wspd", "gust", "wdir", "airt", "relh", "caph"]
CONTINUOUS_COLUMNS = ["hs", "tp", "hmax", "wspd", "gust", "airt", "relh", "caph"]
DIRECTION_COLUMNS = ["wvdir", "wdir"]

# Validation 분할 기준일 (이 시점 이전 데이터만 통계치 산출에 사용)
# Raw time index is UTC+09:00; the split timestamp must be tz-aware too.
TRAIN_SPLIT_DATE = pd.Timestamp("2025-01-01 00:00:00+09:00")

HS_SHORT_MAX_STEPS = 18       # 3h
ATMOS_SHORT_MAX_STEPS = 36    # 6h
G_TO_I_LAG_STEPS = 17         # 2h 50m lag

# ------------------------------------------------------------
# 1.1 Raw 데이터 로드 및 10분 균일 그리드 생성
# ------------------------------------------------------------
print("1.1 Raw 데이터 로드 및 10분 균일 그리드 생성...")
atmos = pd.read_csv(RAW_ATMOS_PATH, parse_dates=[TIME_COL])
wave = pd.read_csv(RAW_WAVE_PATH, parse_dates=[TIME_COL])
train = pd.merge(atmos, wave, on=[STATION_COL, TIME_COL], how="outer")

reindexed_dfs = []
for st, g in train.groupby(STATION_COL):
    g = g.sort_values(TIME_COL).drop_duplicates(subset=[TIME_COL]).set_index(TIME_COL)
    full_idx = pd.date_range(start=g.index.min(), end=g.index.max(), freq=f"{STEP_MINUTES}min", tz=g.index.tz)
    g_re = g.reindex(full_idx)
    g_re[STATION_COL] = st
    reindexed_dfs.append(g_re.reset_index().rename(columns={"index": TIME_COL}))

df = pd.concat(reindexed_dfs, ignore_index=True).sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)

# ------------------------------------------------------------
# 1.2 원본 관측 여부 플래그 저장
# ------------------------------------------------------------
print("1.2 원본 관측 여부 플래그 저장...")
for col in BASE_COLUMNS:
    df[f"{col}_original_observed"] = df[col].notna().astype(np.int8)
    df[f"{col}_imputed"] = np.int8(0)
df["hs_fill_method"] = np.where(df["hs"].notna(), "observed", "missing")

# ------------------------------------------------------------
# 1.3 물리적 이상치 필터링
# ------------------------------------------------------------
print("1.3 물리적 이상치 필터링...")
for col in ["hs", "tp", "hmax"]:
    df.loc[df[col] <= 0, col] = np.nan
for col in ["wspd", "gust"]:
    df.loc[df[col] < 0, col] = np.nan
df.loc[~df["caph"].between(950, 1050), "caph"] = np.nan
df.loc[~df["relh"].between(0, 100), "relh"] = np.nan
for col in DIRECTION_COLUMNS:
    df[col] = df[col] % 360.0

df.loc[df["hmax"].notna() & df["hs"].notna() & (df["hmax"] < df["hs"]), "hmax"] = np.nan
df.loc[df["gust"].notna() & df["wspd"].notna() & (df["gust"] < df["wspd"]), "gust"] = np.nan

# ------------------------------------------------------------
# 1.4 단기 결측 보간 (Causal ffill 중심 or 엄격 제한)
# ------------------------------------------------------------
print("1.4 단기 결측 보간 수행...")
# hs, tp는 미래값 참조를 방지하기 위해 forward fill만 허용
for col in ["hs", "tp"]:
    mask_missing = df[col].isna()
    df[col] = df.groupby(STATION_COL)[col].transform(lambda s: s.ffill(limit=HS_SHORT_MAX_STEPS))
    filled = mask_missing & df[col].notna()
    df.loc[filled, f"{col}_imputed"] = 1
    if col == "hs":
        df.loc[filled, "hs_fill_method"] = "short_ffill"

for col in ["wspd", "gust", "airt", "relh", "caph"]:
    mask_missing = df[col].isna()
    df[col] = df.groupby(STATION_COL)[col].transform(lambda s: s.ffill(limit=ATMOS_SHORT_MAX_STEPS))
    df.loc[mask_missing & df[col].notna(), f"{col}_imputed"] = 1

for col in DIRECTION_COLUMNS:
    df[col] = df.groupby(STATION_COL)[col].transform(lambda s: s.ffill(limit=6) % 360.0)

# ------------------------------------------------------------
# 1.5 I-ORS 장기 결측 복원 (Train 기간만으로 회귀 Fitting)
# ------------------------------------------------------------
print("1.5 I-ORS G-ORS 전이 회귀 복원 (Train Split 기준 학습)...")
hs_wide = df.pivot(index=TIME_COL, columns=STATION_COL, values="hs").sort_index()
obs_wide = df.pivot(index=TIME_COL, columns=STATION_COL, values="hs_original_observed").sort_index()

g_lag = hs_wide["G-ORS"].shift(G_TO_I_LAG_STEPS)
g_obs_lag = obs_wide["G-ORS"].shift(G_TO_I_LAG_STEPS)

# Train 구간에 한정하여 마스킹 구성
train_time_mask = hs_wide.index < TRAIN_SPLIT_DATE
calib_mask = (
    train_time_mask
    & (obs_wide["I-ORS"] == 1)
    & (g_obs_lag == 1)
    & hs_wide["I-ORS"].notna()
    & g_lag.notna()
)

X_calib = g_lag[calib_mask].to_numpy().reshape(-1, 1)
y_calib = hs_wide["I-ORS"][calib_mask].to_numpy()

print(f" -> [Train Only] I-G 동시 유효 캘리브레이션 샘플 수: {len(y_calib)}개")
if len(y_calib) >= 100:
    reg = LinearRegression().fit(X_calib, y_calib)
    print(f" -> I_hs = {reg.coef_[0]:.4f} * G_lag + {reg.intercept_:.4f}")

    i_missing = (df[STATION_COL] == "I-ORS") & df["hs"].isna()
    source_vals = df.loc[i_missing, TIME_COL].map(g_lag).to_numpy()
    
    valid_pred_mask = np.isfinite(source_vals)
    if valid_pred_mask.sum() > 0:
        preds = np.clip(reg.predict(source_vals[valid_pred_mask].reshape(-1, 1)), 0.05, 15.0)
        fill_indices = df.loc[i_missing].index[valid_pred_mask]
        df.loc[fill_indices, "hs"] = preds
        df.loc[fill_indices, "hs_imputed"] = 1
        df.loc[fill_indices, "hs_fill_method"] = "G_lag_transfer"
        print(f" -> I-ORS G 전이 복원 완료: {len(fill_indices)}행")

# Hmax 복원 (Train 구간 데이터만으로 median_ratio 계산)
train_df = df[df[TIME_COL] < TRAIN_SPLIT_DATE]
ratio_train_df = train_df[train_df["hs_original_observed"] == 1]
median_ratio = (ratio_train_df["hmax"] / ratio_train_df["hs"].clip(lower=0.01)).groupby(train_df[STATION_COL]).median()

for st, r in median_ratio.items():
    if pd.isna(r):
        continue
    mask = (df[STATION_COL] == st) & df["hmax"].isna() & df["hs"].notna()
    df.loc[mask, "hmax"] = df.loc[mask, "hs"] * r
    df.loc[mask, "hmax_imputed"] = 1

# ------------------------------------------------------------
# 1.6 기상 변수 공간 편차 전이 (Train 구간 Bias 및 Median만 적용)
# ------------------------------------------------------------
print("1.6 기상 변수 공간 편차 전이 (Train Split 기준)...")
for col in ["wspd", "gust", "caph", "airt", "relh"]:
    wide = df.pivot(index=TIME_COL, columns=STATION_COL, values=col).sort_index()
    train_wide = wide[wide.index < TRAIN_SPLIT_DATE]
    
    for target_st in ["I-ORS", "S-ORS"]:
        if target_st in wide.columns and "G-ORS" in wide.columns:
            bias = (train_wide[target_st] - train_wide["G-ORS"]).dropna().median()
            if np.isnan(bias):
                bias = 0.0
            missing_t = wide[target_st].isna() & wide["G-ORS"].notna()
            if missing_t.sum() > 0:
                wide.loc[missing_t, target_st] = wide.loc[missing_t, "G-ORS"] + bias
    
    # Train 기간 median fallback
    train_medians = train_wide.median()
    wide = wide.fillna(train_medians)

    for st in df[STATION_COL].unique():
        if st in wide.columns:
            st_mask = (df[STATION_COL] == st) & df[col].isna()
            df.loc[st_mask, col] = df.loc[st_mask, TIME_COL].map(wide[st]).to_numpy()
            df.loc[st_mask, f"{col}_imputed"] = 1

for col in DIRECTION_COLUMNS:
    df[col] = df.groupby(STATION_COL)[col].transform(lambda s: s.ffill() % 360.0)

df.to_csv(BASE_V6_PATH, index=False)
print(f"★ 1단계 베이스 V6 저장 완료: {BASE_V6_PATH} (Shape: {df.shape})")

1.1 Raw 데이터 로드 및 10분 균일 그리드 생성...
1.2 원본 관측 여부 플래그 저장...
1.3 물리적 이상치 필터링...
1.4 단기 결측 보간 수행...
1.5 I-ORS G-ORS 전이 회귀 복원 (Train Split 기준 학습)...
 -> [Train Only] I-G 동시 유효 캘리브레이션 샘플 수: 0개
1.6 기상 변수 공간 편차 전이 (Train Split 기준)...
★ 1단계 베이스 V6 저장 완료: train_final_base_v6.csv (Shape: (236304, 33))


In [2]:
# ============================================================
# 2. V6 PHYSICS FEATURE GENERATION (NO BFILL LEAKAGE)
# ============================================================
PHYSICS_V6_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_physics_v6.csv")
df = pd.read_csv(BASE_V6_PATH, parse_dates=[TIME_COL])
df = df.sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)

G = 9.80665
EPS = 1e-6

print("2.1 방향 직교 분해 및 풍향/풍속 벡터...")
wdir_rad = np.deg2rad(df["wdir"])
wvdir_rad = np.deg2rad(df["wvdir"])
df["wdir_sin"] = np.sin(wdir_rad)
df["wdir_cos"] = np.cos(wdir_rad)
df["wvdir_sin"] = np.sin(wvdir_rad)
df["wvdir_cos"] = np.cos(wvdir_rad)
df["u_wind"] = df["wspd"] * np.sin(wdir_rad)
df["v_wind"] = df["wspd"] * np.cos(wdir_rad)
df["u_wave"] = df["hs"] * np.sin(wvdir_rad)
df["v_wave"] = df["hs"] * np.cos(wvdir_rad)
df["wspd2"] = df["wspd"] ** 2
df["wspd3"] = df["wspd"] ** 3
df["gust_minus_wspd"] = df["gust"] - df["wspd"]

print("2.2 풍파 상호작용 및 배열 피처...")
diff = ((df["wdir"] - df["wvdir"] + 180) % 360) - 180
df["wind_wave_diff"] = np.abs(diff)
df["wind_wave_alignment"] = np.cos(np.deg2rad(diff))
df["effective_wind_forcing"] = (df["wspd"] ** 2) * df["wind_wave_alignment"]

print("2.3 시계열 롤링 모멘텀 및 기압 경향...")
groups = df.groupby(STATION_COL, group_keys=False)
df["hs_diff_1h"] = df["hs"] - groups["hs"].shift(6)
df["hs_diff_3h"] = df["hs"] - groups["hs"].shift(18)
df["hs_mean_6h"] = groups["hs"].transform(lambda s: s.rolling(36, min_periods=1).mean())
df["hs_mean_12h"] = groups["hs"].transform(lambda s: s.rolling(72, min_periods=1).mean())
df["hs_max_6h"] = groups["hs"].transform(lambda s: s.rolling(36, min_periods=1).max())
df["hs_max_12h"] = groups["hs"].transform(lambda s: s.rolling(72, min_periods=1).max())

df["wspd_mean_6h"] = groups["wspd"].transform(lambda s: s.rolling(36, min_periods=1).mean())
df["wspd_mean_12h"] = groups["wspd"].transform(lambda s: s.rolling(72, min_periods=1).mean())
df["gust_max_6h"] = groups["gust"].transform(lambda s: s.rolling(36, min_periods=1).max())
df["gust_max_12h"] = groups["gust"].transform(lambda s: s.rolling(72, min_periods=1).max())
df["wspd3_mean_6h"] = groups["wspd"].transform(lambda s: (s ** 3).rolling(36, min_periods=1).mean())

df["caph_change_3h"] = df["caph"] - groups["caph"].shift(18)
df["caph_change_6h"] = df["caph"] - groups["caph"].shift(36)
df["caph_change_12h"] = df["caph"] - groups["caph"].shift(72)

print("2.4 해양 동역학 파생변수...")
df["wave_energy"] = df["hs"] ** 2
df["wave_power"] = (df["hs"] ** 2) * df["tp"]
df["deepwater_wavelength"] = (G * (df["tp"] ** 2)) / (2 * np.pi)
df["wave_steepness"] = df["hs"] / (df["deepwater_wavelength"] + EPS)
df["phase_speed"] = (G * df["tp"]) / (2 * np.pi)
df["wave_age"] = df["phase_speed"] / (df["wspd"] + 1.0)

print("2.5 주기 삼각함수 시간 피처...")
hour = df["time"].dt.hour + df["time"].dt.minute / 60.0
month = df["time"].dt.month
df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
df["month_sin"] = np.sin(2 * np.pi * (month - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (month - 1) / 12)

# ============================================================
# [리키지 차단 핵심]: 무제한 bfill 제거
# inf 치환 후 과거 방향(ffill)만 채우고, 
# 시계열 맨 앞의 lag/shift로 생긴 잔여 NaN은 Train 구간 중앙값(또는 0)으로 대체
# ============================================================
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Causal ffill만 그룹별 적용
df[numeric_cols] = df.groupby(STATION_COL)[numeric_cols].transform(lambda s: s.ffill())

# 맨 첫 관측 이전의 초기 shift NaN 값은 train split 중앙값으로 안전 대치
train_split_mask = df[TIME_COL] < TRAIN_SPLIT_DATE
train_col_medians = df.loc[train_split_mask, numeric_cols].median().fillna(0.0)
df[numeric_cols] = df[numeric_cols].fillna(train_col_medians)

df.to_csv(PHYSICS_V6_PATH, index=False)
print(f"★ 2단계 물리 피처 V6 저장 완료: {PHYSICS_V6_PATH} (Shape: {df.shape})")

2.1 방향 직교 분해 및 풍향/풍속 벡터...
2.2 풍파 상호작용 및 배열 피처...
2.3 시계열 롤링 모멘텀 및 기압 경향...
2.4 해양 동역학 파생변수...
2.5 주기 삼각함수 시간 피처...
★ 2단계 물리 피처 V6 저장 완료: train_final_physics_v6.csv (Shape: (236304, 71))
